In [ ]:
import torch
from torch import nn
import pandas as pd

In [ ]:
# Source:
#    -> https://www.kaggle.com/datasets/mrsimple07/student-exam-performance-prediction/data
# License:
#    -> Apache 2.0

# Loading data as Pandas dataframe

df = pd.read_csv("path_to_file.csv")

In [ ]:
X = torch.tensor(df[["Study Hours", "Previous Exam Score"]].values, dtype=torch.float32)
Y = torch.tensor(df["Pass/Fail"], dtype=torch.float32).reshape((-1, 1))

In [ ]:
model = nn.Sequential(nn.Linear(2, 10), nn.ReLU(), nn.Linear(10, 1))

loss_fn = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_entries = X.size(0)
batch_size = 32


for i in range(0, 2000):
    loss_sum = 0
    # packing the training dataset entries into batch size of 32 (mini batch learning) to avoid
    # loading the entire dataset examples into memory during training
    for start in range(0, num_entries, batch_size):
        end = min(num_entries, start + batch_size)
        X_data = X[start:end]
        Y_data = Y[start:end]

        optimizer.zero_grad()
        outputs = model(X_data)
        loss = loss_fn(outputs, Y_data)
        loss.backward()
        loss_sum += loss.item()
        optimizer.step()
    # tracking the loss after each 10 epochs
    if i % 10 == 0:
        print(loss_sum)

In [ ]:
# Evaluating the model

model.eval()
with torch.no_grad():
    outputs = model(X)
    Y_pred = nn.functional.sigmoid(outputs) > 0.5
    Y_pred_correct = Y_pred.type(torch.float32) == Y
    print(Y_pred_correct.type(torch.float32).mean())